# QLoRA fine-tuning — Gemma 4 E2B (Hungarian text→JSON extraction)

Thin Colab wrapper around `src/train.py` (dev-plan step 3.4). All the actual
logic (data loading, model/LoRA setup, `SFTTrainer` config) lives in the
script, not here — this notebook just installs Unsloth, fetches the repo +
data, and invokes the CLI.

**Runtime:** Runtime → Change runtime type → T4 GPU, before running anything below.

In [ ]:
%%capture
!pip install unsloth

## Repo

Clone the repo (set `REPO_URL` below) so `src/` is available.

In [ ]:
REPO_URL = "https://github.com/<your-username>/hu-llm-finetune-lora.git"  # noqa: set this
REPO_DIR = "hu-llm-finetune-lora"

!git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

## Synthetic training data

`data/synthetic/` is gitignored (regenerable, LLM-generated — see `CLAUDE.md`),
so it doesn't come with the clone. Upload the three domain JSONL files
(`medical.jsonl`, `business.jsonl`, `technology.jsonl`) produced locally by
`python src/generate_data.py`, e.g. via a zip:

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # select a zip of data/synthetic/*.jsonl
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall("data/synthetic")

## Smoke test

Quick end-to-end shakeout (~60 steps) before committing to a full run.

In [ ]:
!python src/train.py --smoke --out outputs/smoke

## Full training run

In [ ]:
!python src/train.py --out outputs/adapter

The trained LoRA adapter + tokenizer are saved to `outputs/adapter/`.
Download it (or push it to the Hugging Face Hub — step 3.7, not part of this
notebook) before the Colab runtime recycles.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("adapter", "zip", "outputs/adapter")
files.download("adapter.zip")